# Qwen3-TTS Local Voice Cloning for NVIDIA RTX A5000

This notebook is focused on one task only: cloning **your own voice locally** with **Qwen3-TTS-12Hz-1.7B-Base**, which is the best-quality cloning model from the tutorial.

What is different from the original tutorial:
- Uses the **1.7B Base** model only.
- Assumes **local execution** on your Linux machine with an **NVIDIA RTX A5000**.
- Adds **GPU-aware defaults** for precision, attention backend, TF32, and cache paths.
- Uses a **local reference audio file only**. No URLs and no Colab flow.
- Adds **reference-audio cleanup** to improve cloning quality.
- Reuses the extracted clone prompt so you can generate multiple lines from the same voice quickly.

Reference audio recommendations:
- Use a **clean WAV** with one speaker only.
- Ideal duration: **5 to 12 seconds**.
- Avoid music, reverb, clipping, and long silences.
- Provide the **exact transcript** of the reference audio.

In [ ]:
# This project uses a uv-managed virtual environment.
# Install or refresh dependencies from the terminal with: uv sync

import importlib.metadata as metadata

required_packages = ["qwen-tts", "accelerate", "soundfile", "librosa"]
for package_name in required_packages:
    print(f"{package_name}: {metadata.version(package_name)}")

print("uv environment is ready for the remaining notebook cells.")

# Optional GPU speed-up outside the notebook, if you want it later:
# uv pip install --python .venv/bin/python flash-attn --no-build-isolation

/home/roderickperez/DataScienceProjects/EAGE_Python_course/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/roderickperez/DataScienceProjects/EAGE_Python_course/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import importlib.util
import os
from pathlib import Path

os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")  # 5-min timeout per chunk
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import librosa
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUTPUT_DIR = Path("tts_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
bf16_supported = device == "cuda" and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
dtype = torch.bfloat16 if bf16_supported else (torch.float16 if device == "cuda" else torch.float32)
flash_attn_available = importlib.util.find_spec("flash_attn") is not None
attn_implementation = "flash_attention_2" if device == "cuda" and flash_attn_available else ("sdpa" if device == "cuda" else None)

if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"Device: {device}")
print(f"GPU: {gpu_name}")
print(f"dtype: {dtype}")
print(f"Attention backend: {attn_implementation}")
if device == "cuda":
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    print("A5000 note: 1.7B inference should fit comfortably for single-speaker local cloning.")
else:
    print("CUDA was not detected. Local cloning will still work, but it will be much slower.")

ModuleNotFoundError: No module named 'librosa'

In [ ]:
def play_audio(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Audio file not found: {file_path}")
    display(Audio(str(file_path)))


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def prepare_reference_audio(input_path, output_path=OUTPUT_DIR / "reference_clean.wav", trim_db=30):
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(
            f"Reference audio not found: {input_path}. Update REFERENCE_AUDIO before running this cell."
        )

    audio, sample_rate = librosa.load(input_path, sr=None, mono=True)
    trimmed_audio, _ = librosa.effects.trim(audio, top_db=trim_db)

    if trimmed_audio.size == 0:
        raise ValueError("Reference audio is empty after trimming. Use a cleaner sample or lower trim_db.")

    peak = np.max(np.abs(trimmed_audio))
    if peak > 0:
        trimmed_audio = 0.98 * (trimmed_audio / peak)

    duration = len(trimmed_audio) / sample_rate
    if duration < 3:
        print(f"Warning: reference duration is only {duration:.2f}s. Aim for at least 5s for better identity capture.")
    elif duration > 15:
        print(f"Warning: reference duration is {duration:.2f}s. Shorter clips, around 5-12s, usually clone more cleanly.")

    sf.write(output_path, trimmed_audio, sample_rate)
    print(f"Prepared reference saved to: {output_path}")
    print(f"Sample rate: {sample_rate} Hz | Duration: {duration:.2f} s")
    return output_path


def save_clone(model, voice_clone_prompt, text, language, output_path):
    wavs, sample_rate = model.generate_voice_clone(
        text=text,
        language=language,
        voice_clone_prompt=voice_clone_prompt,
    )
    output_path = Path(output_path)
    sf.write(output_path, wavs[0], sample_rate)
    print(f"Saved cloned audio to: {output_path}")
    return output_path

In [ ]:
from huggingface_hub import snapshot_download

MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

# Step 1: download / verify model files with resume support
print(f"Checking / downloading model: {MODEL_NAME}")
print(f"Cache: {os.environ.get('HF_HOME', '~/.cache/huggingface')}")
model_path = snapshot_download(
    repo_id=MODEL_NAME,
    resume_download=True,
    ignore_patterns=["*.msgpack", "*.h5"],  # skip TF/Flax weights
)
print(f"Model files at: {model_path}")

# Step 2: build load kwargs and load from local cache (no network access needed)
load_kwargs = {
    "device_map": {"" : f"cuda:{torch.cuda.current_device()}"} if device == "cuda" else None,
    "dtype": dtype,
}
if attn_implementation is not None:
    load_kwargs["attn_implementation"] = attn_implementation

load_kwargs = {key: value for key, value in load_kwargs.items() if value is not None}

clear_gpu_memory()
print("Loading model from local cache...")
base_model = Qwen3TTSModel.from_pretrained(model_path, **load_kwargs)
print("Voice cloning model loaded.")

## Configure Your Local Reference Voice

Update the two variables below before running the cell:
- `REFERENCE_AUDIO`: path to a local WAV with your voice.
- `REFERENCE_TRANSCRIPT`: the exact spoken words in that file.

The transcript quality matters. If the transcript is wrong, the cloned result usually gets worse.

In [ ]:
REFERENCE_AUDIO = "data/my_voice_reference.wav"
REFERENCE_TRANSCRIPT = "Hola, esta es una muestra corta de mi voz grabada en un ambiente silencioso para clonar mi identidad vocal de la forma mas fiel posible."

prepared_reference = prepare_reference_audio(REFERENCE_AUDIO)
play_audio(prepared_reference)

## Extract The Voice Clone Prompt Once

This is the expensive identity-extraction step. Run it once for your reference sample, then reuse the resulting prompt for many generations.

In [ ]:
voice_clone_prompt = base_model.create_voice_clone_prompt(
    ref_audio=str(prepared_reference),
    ref_text=REFERENCE_TRANSCRIPT,
)
print("Voice clone prompt created successfully.")

## Generate One Local Clone

Start with one sentence, listen to the result, then iterate on your reference clip if needed.

In [ ]:
TARGET_TEXT = "Hola, esta es una prueba local de clonacion de voz usando Qwen3-TTS y mi GPU NVIDIA RTX A5000."
TARGET_LANGUAGE = "Spanish"
OUTPUT_FILE = OUTPUT_DIR / "clone_spanish.wav"

generated_path = save_clone(
    model=base_model,
    voice_clone_prompt=voice_clone_prompt,
    text=TARGET_TEXT,
    language=TARGET_LANGUAGE,
    output_path=OUTPUT_FILE,
)
play_audio(generated_path)

## Generate Multiple Sentences With The Same Voice

Once the prompt is extracted, you can batch several test lines without rebuilding the voice identity.

In [ ]:
samples = [
    ("Spanish", "Este es otro ejemplo para comprobar si la entonacion y el timbre se parecen a mi voz real.", OUTPUT_DIR / "clone_spanish_2.wav"),
    ("English", "This is a second sample to verify how stable the cloned voice remains across languages.", OUTPUT_DIR / "clone_english.wav"),
]

for language, text, output_path in samples:
    saved_file = save_clone(
        model=base_model,
        voice_clone_prompt=voice_clone_prompt,
        text=text,
        language=language,
        output_path=output_path,
    )
    print(f"Previewing: {saved_file.name}")
    play_audio(saved_file)

## Quality Tuning Notes

If the result is not close enough to your real voice, adjust these first:
1. Replace the reference clip with a cleaner one.
2. Make sure the reference transcript is exact.
3. Use a neutral speaking style in the reference audio.
4. Keep the reference clip short and focused on one speaker.
5. Regenerate after removing breaths, room echo, or long pauses from the source audio.

A5000-specific note:
- Stay with the **1.7B Base** model for best quality.
- If `flash-attn` installs successfully, it should be the fastest option here.
- If it does not install, the notebook falls back to `sdpa`, which is usually still fine on your GPU.

In [ ]:
# Optional cleanup when you are done.
# del base_model
# clear_gpu_memory()